In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Load the datasets
train_df = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test_df = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")

# Display the first 5 rows of the training data
print(train_df.head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Clean up dataset: replace inf/-inf with NaN
train_df = train_df.replace([np.inf, -np.inf], np.nan)

# Basic dataset overview
print("=== Data Info ===")
train_df.info()

print("\n=== Data Description ===")
print(train_df.describe())

# 1) Histogram of SalePrice (raw and log-transformed)
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(train_df['SalePrice'].dropna(), bins=30, kde=True)
plt.title("Distribution of SalePrice")
plt.xlabel("SalePrice")
plt.ylabel("Frequency")

plt.subplot(1, 2, 2)
sns.histplot(np.log1p(train_df['SalePrice'].dropna()), bins=30, kde=True, color="orange")
plt.title("Distribution of Log(SalePrice)")
plt.xlabel("Log(SalePrice)")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

# 2) Heatmap of correlations (top 10 numeric features related to SalePrice)
corr_matrix = train_df.corr(numeric_only=True)
top_corr_features = corr_matrix['SalePrice'].abs().sort_values(ascending=False).head(11).index
plt.figure(figsize=(10, 8))
sns.heatmap(train_df[top_corr_features].corr(), annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation Heatmap of Top 10 Features with SalePrice")
plt.show()

# 3) Boxplots for categorical variable (Neighborhood vs SalePrice)
plt.figure(figsize=(15, 8))
sns.boxplot(x="Neighborhood", y="SalePrice", data=train_df)
plt.xticks(rotation=45)
plt.title("SalePrice by Neighborhood")
plt.xlabel("Neighborhood")
plt.ylabel("SalePrice")
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

# Separate features (X) and target (y)
X = train_df.drop("SalePrice", axis=1)
y = train_df["SalePrice"]

# Perform 80/20 train-test split
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set shape:", X_train.shape)
print("Validation set shape:", X_valid.shape)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# Train a simple Linear Regression model
model = LinearRegression()
model.fit(X_train.select_dtypes(include=[np.number]).fillna(0), y_train)

# Make predictions on validation set
y_pred = model.predict(X_valid.select_dtypes(include=[np.number]).fillna(0))

# Calculate metrics
r2 = r2_score(y_valid, y_pred)
mse = mean_squared_error(y_valid, y_pred)
rmse = np.sqrt(mse)

print(f"R^2 Score: {r2:.4f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")

In [ ]:
# --- Feature Engineering ---
train_fe = train_df.copy()

# Total square footage
train_fe["TotalSF"] = (
    train_fe["TotalBsmtSF"].fillna(0)
    + train_fe["1stFlrSF"].fillna(0)
    + train_fe["2ndFlrSF"].fillna(0)
)

# Total bathrooms
train_fe["TotalBath"] = (
    train_fe["FullBath"].fillna(0)
    + 0.5 * train_fe["HalfBath"].fillna(0)
    + train_fe["BsmtFullBath"].fillna(0)
    + 0.5 * train_fe["BsmtHalfBath"].fillna(0)
)

# House age
train_fe["HouseAge"] = train_fe["YrSold"] - train_fe["YearBuilt"]

# Years since remodel
train_fe["RemodAge"] = train_fe["YrSold"] - train_fe["YearRemodAdd"]

# Combined quality score
train_fe["OverallScore"] = train_fe["OverallQual"] * train_fe["OverallCond"]

# --- Split Features and Target ---
X = train_fe.drop("SalePrice", axis=1)
y = train_fe["SalePrice"]

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Keep only numeric features & fill NA
X_train_num = X_train.select_dtypes(include=[np.number]).fillna(0)
X_valid_num = X_valid.select_dtypes(include=[np.number]).fillna(0)

# --- Train Model ---
model = LinearRegression()
model.fit(X_train_num, y_train)

# --- Predictions ---
y_pred = model.predict(X_valid_num)

# --- Metrics ---
r2 = r2_score(y_valid, y_pred)
mse = mean_squared_error(y_valid, y_pred)
rmse = np.sqrt(mse)

print(f"R^2 Score: {r2:.4f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor

# --- Feature Engineering ---
train_fe = train_df.copy()

train_fe["TotalSF"] = (
    train_fe["TotalBsmtSF"].fillna(0)
    + train_fe["1stFlrSF"].fillna(0)
    + train_fe["2ndFlrSF"].fillna(0)
)

train_fe["TotalBath"] = (
    train_fe["FullBath"].fillna(0)
    + 0.5 * train_fe["HalfBath"].fillna(0)
    + train_fe["BsmtFullBath"].fillna(0)
    + 0.5 * train_fe["BsmtHalfBath"].fillna(0)
)

train_fe["HouseAge"] = train_fe["YrSold"] - train_fe["YearBuilt"]
train_fe["RemodAge"] = train_fe["YrSold"] - train_fe["YearRemodAdd"]
train_fe["OverallScore"] = train_fe["OverallQual"] * train_fe["OverallCond"]

# --- Split Features & Target ---
X = train_fe.drop("SalePrice", axis=1)
y = train_fe["SalePrice"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Keep only numeric features
X_train_num = X_train.select_dtypes(include=[np.number]).fillna(0)
X_valid_num = X_valid.select_dtypes(include=[np.number]).fillna(0)

# --- Helper to train & evaluate ---
def evaluate_model(model, X_train, y_train, X_valid, y_valid):
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    rmse = np.sqrt(mean_squared_error(y_valid, preds))
    return rmse, model

# --- Train Models ---
results = {}

rf = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)
gb = GradientBoostingRegressor(n_estimators=500, learning_rate=0.05, random_state=42)
xgb = XGBRegressor(n_estimators=500, learning_rate=0.05, random_state=42, n_jobs=-1)

results["RandomForest"], rf_model = evaluate_model(rf, X_train_num, y_train, X_valid_num, y_valid)
results["GradientBoosting"], gb_model = evaluate_model(gb, X_train_num, y_train, X_valid_num, y_valid)
results["XGBoost"], xgb_model = evaluate_model(xgb, X_train_num, y_train, X_valid_num, y_valid)

# --- Print Validation RMSE ---
print("Validation RMSE:")
for model_name, rmse in results.items():
    print(f"{model_name}: {rmse:.2f}")

# --- Feature Importance Plots ---
def plot_feature_importance(model, X, title):
    importances = pd.Series(model.feature_importances_, index=X.columns)
    top10 = importances.sort_values(ascending=False).head(10)
    
    plt.figure(figsize=(8, 5))
    sns.barplot(x=top10.values, y=top10.index, palette="viridis")
    plt.title(f"Top 10 Feature Importances - {title}")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.show()

plot_feature_importance(rf_model, X_train_num, "Random Forest")
plot_feature_importance(gb_model, X_train_num, "Gradient Boosting")
plot_feature_importance(xgb_model, X_train_num, "XGBoost")

In [ ]:
# --- 1) Model RMSE Comparison ---
rmse_df = pd.DataFrame(list(results.items()), columns=["Model", "RMSE"])

plt.figure(figsize=(7, 5))
sns.barplot(x="Model", y="RMSE", data=rmse_df, palette="mako")
plt.title("Validation RMSE by Model")
plt.ylabel("RMSE (Lower is Better)")
plt.xlabel("Model")
plt.show()

# --- 2) Residual Plots ---
def plot_residuals(model, X_valid, y_valid, title):
    preds = model.predict(X_valid)
    residuals = y_valid - preds
    
    plt.figure(figsize=(7, 5))
    sns.scatterplot(x=preds, y=residuals, alpha=0.6)
    plt.axhline(0, color="red", linestyle="--")
    plt.title(f"Residual Plot - {title}")
    plt.xlabel("Predicted SalePrice")
    plt.ylabel("Residuals (Actual - Predicted)")
    plt.show()

plot_residuals(rf_model, X_valid_num, y_valid, "Random Forest")
plot_residuals(gb_model, X_valid_num, y_valid, "Gradient Boosting")
plot_residuals(xgb_model, X_valid_num, y_valid, "XGBoost")

# --- 3) Prediction vs Actual Plot ---
def plot_pred_vs_actual(model, X_valid, y_valid, title):
    preds = model.predict(X_valid)
    
    plt.figure(figsize=(7, 5))
    sns.scatterplot(x=y_valid, y=preds, alpha=0.6)
    plt.plot([y_valid.min(), y_valid.max()],
             [y_valid.min(), y_valid.max()],
             color="red", lw=2)  # reference line
    plt.title(f"Predicted vs Actual - {title}")
    plt.xlabel("Actual SalePrice")
    plt.ylabel("Predicted SalePrice")
    plt.show()

plot_pred_vs_actual(rf_model, X_valid_num, y_valid, "Random Forest")
plot_pred_vs_actual(gb_model, X_valid_num, y_valid, "Gradient Boosting")
plot_pred_vs_actual(xgb_model, X_valid_num, y_valid, "XGBoost")

# --- 4) Feature Impact with SHAP (optional if installed) ---
try:
    import shap
    explainer = shap.TreeExplainer(xgb_model)
    shap_values = explainer.shap_values(X_valid_num)
    
    shap.summary_plot(shap_values, X_valid_num, plot_type="bar", max_display=10)
    shap.summary_plot(shap_values, X_valid_num, max_display=10)  # beeswarm plot
except ImportError:
    print("SHAP not installed. Skipping feature impact plots.")

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor

# --- Feature Engineering on test set (same as train) ---
test_fe = test_df.copy()

test_fe["TotalSF"] = (
    test_fe["TotalBsmtSF"].fillna(0)
    + test_fe["1stFlrSF"].fillna(0)
    + test_fe["2ndFlrSF"].fillna(0)
)

test_fe["TotalBath"] = (
    test_fe["FullBath"].fillna(0)
    + 0.5 * test_fe["HalfBath"].fillna(0)
    + test_fe["BsmtFullBath"].fillna(0)
    + 0.5 * test_fe["BsmtHalfBath"].fillna(0)
)

test_fe["HouseAge"] = test_fe["YrSold"] - test_fe["YearBuilt"]
test_fe["RemodAge"] = test_fe["YrSold"] - test_fe["YearRemodAdd"]
test_fe["OverallScore"] = test_fe["OverallQual"] * test_fe["OverallCond"]

# --- Prepare full training data ---
X_full = train_fe.drop("SalePrice", axis=1)
y_full = train_fe["SalePrice"]

# Keep only numeric features in both train and test
X_full_num = X_full.select_dtypes(include=[np.number]).fillna(0)
test_num = test_fe.select_dtypes(include=[np.number]).fillna(0)

# Align columns (important for Kaggle test set!)
X_full_num, test_num = X_full_num.align(test_num, join="left", axis=1, fill_value=0)

# --- Train best model (XGBoost) ---
best_model = XGBRegressor(
    n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42, n_jobs=-1
)
best_model.fit(X_full_num, y_full)

# --- Predict on test set ---
test_pred = best_model.predict(test_num)

# --- Create submission file ---
submission = pd.DataFrame({
    "Id": test_df["Id"],
    "SalePrice": test_pred
})

submission.to_csv("submission.csv", index=False)
print("submission.csv file created!")